<a href="https://colab.research.google.com/github/gilbert-233/Baruch-CIS-3120-FTRA-git-demo/blob/main/Notebook1_TMDb_Acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import and keys
Import the libraries we need (`requests` for API calls, `pandas` for tables, `json`/`os` for caching) and load the TMDb API key from Colab Secrets so it stays out of the source code.


In [ ]:
import requests, time, json, os
import pandas as pd
from google.colab import userdata

API_KEY = userdata.get("TMDB_API_KEY")
BASE = "https://api.themoviedb.org/3"

API Helper Function
`tmdb_get` is the single function every TMDb request goes through. It adds the API key, retries up to 3 times with increasing waits if the request fails or hits the rate limit (HTTP 429), and pauses briefly between calls.

In [ ]:
def tmdb_get(path, retries=3, **params):
    for attempt in range(retries):
        try:
            r = requests.get(f"{BASE}{path}",
                             params={"api_key": API_KEY, **params}, timeout=10)
            if r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            time.sleep(0.1)
            return r.json()
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return None

Collect movieID
This function uses the `/discover/movie` endpoint to find movies released in each year  choosen , sorted by revenue. Sorting by revenue favors movies that have real budget and revenue data, since many TMDb entries list 0. Duplicate IDs are removed so each movie is fetched only once.

In [ ]:
def fetch_movie_ids(years, pages_per_year=5):
    ids = []
    for year in years:
        for page in range(1, pages_per_year + 1):
            data = tmdb_get("/discover/movie", primary_release_year=year,
                            sort_by="revenue.desc", page=page)
            if data:
                ids += [m["id"] for m in data["results"]]
    return list(dict.fromkeys(ids))

Fetch details
The list endpoints return only basic information, so `fetch_details` calls `/movie/{movie_id}` for each movie to get budget, revenue, runtime, language, genres, and production countries. Each response is saved to a local JSON cache, so if the notebook stops or is rerun, movies already downloaded are not requested again. This protects against rate limits and saves time.

In [ ]:
CACHE = "tmdb_details_cache.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def fetch_details(movie_id):
    key = str(movie_id)
    if key not in cache:
        cache[key] = tmdb_get(f"/movie/{movie_id}")
        json.dump(cache, open(CACHE, "w"))
    return cache[key]

Building the Raw DataFrame
`build_df` keeps only the fields the team needs and reshapes genres and production countries into simple lists . The result, `df_raw`, is the unmodified raw data handed to the cleaning step.

In [ ]:
def build_df(ids):
    rows = []
    for i in ids:
        d = fetch_details(i)
        if not d:
            continue
        rows.append({
            "movie_id": d["id"], "title": d["title"],
            "release_date": d["release_date"], "budget": d["budget"],
            "revenue": d["revenue"], "runtime": d["runtime"],
            "original_language": d["original_language"],
            "genres": [g["name"] for g in d["genres"]],
            "production_countries": [c["iso_3166_1"] for c in d["production_countries"]],
        })
    return pd.DataFrame(rows)

df_raw = build_df(fetch_movie_ids(range(2015, 2025), 5))
df_raw.head()

,movie_id,title,release_date,budget,revenue,runtime,original_language,genres,production_countries
0,140607,Star Wars: The Force Awakens,2015-12-15,245000000,2068223624,136,en,"[Adventure, Action, Science Fiction]",[US]
1,135397,Jurassic World,2015-06-06,150000000,1671537444,124,en,"[Adventure, Science Fiction, Thriller]",[US]
2,168259,Furious 7,2015-04-01,190000000,1515400000,138,en,"[Action, Crime, Thriller]",[US]
3,99861,Avengers: Age of Ultron,2015-04-22,235000000,1405403694,141,en,"[Action, Adventure, Science Fiction]",[US]
4,211672,Minions,2015-06-17,74000000,1159457503,91,en,"[Family, Animation, Adventure, Comedy]",[US]


 Schema Check
`info()` shows the columns, data types, and non-null counts, and the second line counts movies where budget or revenue is 0. In TMDb, 0 means the value is missing, not that the movie had no budget. These rows will be handled in the cleaning step.

In [ ]:
df_raw.info()
print((df_raw[["budget", "revenue"]] == 0).sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   movie_id              1000 non-null   int64 
 1   title                 1000 non-null   object
 2   release_date          1000 non-null   object
 3   budget                1000 non-null   int64 
 4   revenue               1000 non-null   int64 
 5   runtime               1000 non-null   int64 
 6   original_language     1000 non-null   object
 7   genres                1000 non-null   object
 8   production_countries  1000 non-null   object
dtypes: int64(4), object(5)
memory usage: 70.4+ KB
budget     154
revenue      0
dtype: int64


Test Pull : 80 movies ,9 columns , no nulls .
 Budget zeros:8   Revenue zeros: 8
release_date is a string; genres/production_countries are lists.

In [ ]:
df_raw.sample(5)

,movie_id,title,release_date,budget,revenue,runtime,original_language,genres,production_countries
427,290859,Terminator: Dark Fate,2019-10-23,185000000,261119292,128,en,"[Science Fiction, Action, Adventure, Thriller]",[US]
102,127380,Finding Dory,2016-06-16,200000000,1029266989,97,en,"[Adventure, Animation, Family]",[US]
165,270774,Skiptrace,2016-07-21,32000000,136579287,107,en,"[Action, Adventure, Crime, Comedy]","[CN, HK, US]"
417,530915,1917,2019-12-25,100000000,446064352,119,en,"[War, Drama, History]","[US, GB, ES, IN]"
514,748685,The Sacrifice,2020-10-23,0,161047608,122,zh,"[War, History, Action, Drama]",[CN]
